# Практика № 6 
## Вариант 7


In [1]:
%pip install pandas openpyxl

##  Загрузка данных


In [2]:
import pandas as pd

files = ["data/orders_2024_01.csv", "data/orders_2024_02.csv", "data/orders_2024_03.csv"]
orders = pd.concat([pd.read_csv(f, parse_dates=["order_datetime"]) for f in files],
                   ignore_index=True)
sheets = pd.read_excel("data/products.xlsx", sheet_name=["products", "categories"])
products = sheets["products"]
categories = sheets["categories"]

first = pd.read_json("data/events_records.json").T
first.columns = ["event_id", "user_id", "event_time", "event_type", "page"]
second = pd.read_json("data/events_lines.json", lines=True)
events = pd.concat([first, second], ignore_index=True)
events["user_id"] = events["user_id"].astype("int64")
events["event_time"] = pd.to_datetime(events["event_time"])
print(f"Заказов: {len(orders)}, товаров: {len(products)}, событий: {len(events)}")
display(orders.head())
display(events.head())

Заказов: 15000, товаров: 30, событий: 4000


,order_id,user_id,order_datetime,product_id,price,quantity,status
0,1,1044,2024-01-03 18:39:26,26,14038.15,1,refunded
1,2,1079,2024-01-03 17:45:43,16,9172.60,1,paid
2,3,1041,2024-01-06 22:46:38,25,9035.26,2,paid
3,4,1083,2024-01-08 21:03:51,9,15234.33,3,paid
4,5,1068,2024-01-03 23:26:53,24,4134.38,2,paid


,event_id,user_id,event_time,event_type,page
0,1,1085,2024-01-30 20:37:32,page_view,/product/26
1,2,1006,2024-02-24 22:55:13,add_to_cart,/product/28
2,3,1024,2024-02-14 17:49:43,click,/product/27
3,4,1062,2024-02-16 11:21:32,click,/product/1
4,5,1012,2024-02-08 05:25:21,add_to_cart,/


## 1 Объединение и признаки пользователей


In [3]:
catalog = products.drop(columns="category_name").merge(
    categories, on="category_id", how="left", validate="many_to_one")
orders = orders.merge(catalog, on="product_id", how="left", validate="many_to_one")
assert orders["category_name"].notna().all()
assert events["event_id"].is_unique

orders["paid_revenue"] = (orders["price"] * orders["quantity"]).where(
    orders["status"].eq("paid"), 0)
order_features = orders.groupby("user_id").agg(
    orders_count=("order_id", "nunique"),
    revenue_sum=("paid_revenue", "sum"),
    first_order=("order_datetime", "min"))
event_features = events.groupby("user_id").agg(
    events_total=("event_id", "size"),
    first_event=("event_time", "min"))

users = order_features.join(event_features, how="outer")
features = ["orders_count", "revenue_sum", "events_total"]
users[features] = users[features].fillna(0)
users[["orders_count", "events_total"]] = users[["orders_count", "events_total"]].astype(int)
users["first_activity"] = users[["first_order", "first_event"]].min(axis=1)
assert users["first_activity"].notna().all()
users = users[features + ["first_activity"]]
display(users.head())
print("Всего пользователей:", len(users))

,orders_count,revenue_sum,events_total,first_activity
user_id,,,,
1001,177,3539635.84,54,2024-01-01 06:25:31
1002,123,2196505.04,41,2024-01-02 12:57:34
1003,156,3400406.47,39,2024-01-02 02:02:44
1004,147,3313341.75,37,2024-01-01 00:19:34
1005,141,2484426.17,44,2024-01-01 01:11:51


Всего пользователей: 100


## 2 Разделение по времени первой активности
Разделить пользователей на train/test по времени первой активности.

In [6]:
users = users.sort_values("first_activity")
cutoff = users["first_activity"].iloc[int(len(users) * 0.8)]
train = users.loc[users["first_activity"] < cutoff].copy()
test = users.loc[users["first_activity"] >= cutoff].copy()
assert len(train) > 0 and len(test) > 0
assert train.index.intersection(test.index).empty
assert len(train) + len(test) == len(users)
assert train["first_activity"].max() < test["first_activity"].min()
print("Граница:", cutoff)
print(f"Train: {len(train)} пользователей; test: {len(test)} пользователей")
print("Первая активность train:", train.first_activity.min(), "—", train.first_activity.max())
print("Первая активность test:", test.first_activity.min(), "—", test.first_activity.max())

Граница: 2024-01-01 22:24:30
Train: 80 пользователей; test: 20 пользователей
Первая активность train: 2024-01-01 00:10:25 — 2024-01-01 21:57:18
Первая активность test: 2024-01-01 22:24:30 — 2024-01-05 18:26:42


## 3 Сравнение распределений
Сравнить распределения ключевых признаков (orders_count, revenue_sum, events_total) между
train и test

In [7]:
for feature in features:
    print("Признак:", feature)
    comparison = pd.DataFrame({"train": train[feature].describe(),
                               "test": test[feature].describe()})
    display(comparison.round(2))

means = pd.DataFrame({"train_mean": train[features].mean(),
                      "test_mean": test[features].mean()})
means["difference_percent"] = (means["test_mean"] / means["train_mean"] - 1) * 100
display(means.round(2))

Признак: orders_count


,train,test
count,80.00,20.00
mean,150.31,148.75
std,13.13,11.71
min,122.00,123.00
25%,141.00,138.00
50%,152.50,152.50
75%,161.00,157.00
max,177.00,166.00


Признак: revenue_sum


,train,test
count,80.00,20.00
mean,3092019.08,3023373.38
std,366145.04,380644.40
min,2260785.55,2196505.04
25%,2833203.50,2727204.58
50%,3125580.04,3076913.26
75%,3379822.19,3364500.05
max,3916197.05,3538363.46


Признак: events_total


,train,test
count,80.00,20.00
mean,39.67,41.30
std,7.50,6.83
min,24.00,33.00
25%,34.00,36.75
50%,39.00,40.50
75%,43.25,43.00
max,64.00,59.00


,train_mean,test_mean,difference_percent
orders_count,150.31,148.75,-1.04
revenue_sum,3092019.08,3023373.38,-2.22
events_total,39.67,41.30,4.10


## 4 Вывод о «справедливости» разбиения
Сделать вывод, является ли разбиение по времени «справедливым».

In [8]:
for feature, row in means.iterrows():
    print(f"{feature}: среднее test отличается от train на {row['difference_percent']:+.2f}%")
train.to_csv("train_users.csv", encoding="utf-8-sig")
test.to_csv("test_users.csv", encoding="utf-8-sig")
print("Группы сохранены: train_users.csv и test_users.csv")

orders_count: среднее test отличается от train на -1.04%
revenue_sum: среднее test отличается от train на -2.22%
events_total: среднее test отличается от train на +4.10%
Группы сохранены: train_users.csv и test_users.csv


**Интерпретация.** Более поздние пользователи могли иметь меньше времени для накопления заказов и событий. Поэтому различие признаков может объясняться не только активностью, но и длительностью наблюдения. Test также меньше train, и его оценки менее устойчивы.

Это разделение **по времени первой активности пользователей**, а не разделение всех записей по общей дате. Признаки здесь рассчитаны за весь доступный период для описательного сравнения. Использовать их без изменений для прогноза на дату первой активности нельзя: они содержат будущие относительно этой даты действия. Для ML нужно определить дату прогноза и одинаковое окно истории, а признаки считать только по доступным к этой дате данным.

Для окончательного вывода на предоставленных данных см. следующую ячейку.

### Итог для предоставленных данных
Получено **80 пользователей в train и 20 в test**. Временная граница — **1 января 2024 года, 22:24:30**.

Средние значения test относительно train:
- `orders_count`: ниже на **1,04%**;
- `revenue_sum`: ниже на **2,22%**;
- `events_total`: выше на **4,10%**.

Средние значения близки, выраженного сдвига по ним не видно. Это позволяет считать разбиение **условно сопоставимым по выбранным признакам**. Но одинаковость распределений не доказана: нужно учитывать квартили, разброс и малый размер test. Для временной проверки новых пользователей подход оправдан, для полноценной ML-оценки необходимы ограничения по окну наблюдения, описанные выше.

